# Cross-Industry Accelerator — Data Ingestion
### Auto-Discover & Validate CSV Data Sources

This notebook **ingests raw CSV data** into the Fabric Lakehouse Files area and performs
schema validation, profiling, and data quality checks before loading to Lakehouse/Warehouse/Eventhouse.

**What it does:**
1. Runs `00_Industry_Config` to pick up industry settings and auto-discovered tables
2. Reads every CSV file, infers schema, counts rows, identifies data types
3. Profiles each table (nulls, distinct counts, min/max for numeric columns)
4. Generates a data quality report with issues flagged
5. Produces a full data catalog (table → columns → types → stats)

> **Prerequisites:**
> 1. Upload your industry CSV files to `/lakehouse/default/Files/<industry>_data/`
> 2. Attach a **Lakehouse** to this notebook
> 3. Set the `INDUSTRY` variable in `00_Industry_Config`

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RUN CONFIGURATION NOTEBOOK
# ════════════════════════════════════════════════════════════════════════

In [ ]:
%run 00_Industry_Config

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# START PIPELINE STEP TRACKING
# ════════════════════════════════════════════════════════════════════════

try:
    pipeline_step_start("01_Data_Ingestion", "Profile schemas, detect quality issues, validate data")
except NameError:
    pass

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# START PIPELINE STEP TRACKING
# ════════════════════════════════════════════════════════════════════════

try:
    pipeline_step_start("01_Data_Ingestion", "Profile schemas, detect quality issues, validate data")
except NameError:
    pass

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# FULL SCHEMA DISCOVERY — Read all CSVs and build catalog (ZT-hardened)
# ════════════════════════════════════════════════════════════════════════

from pyspark.sql import functions as F
from pyspark.sql.types import NumericType, StringType, TimestampType, DateType
import pandas as pd

ALL_DISCOVERED = DIM_TABLES + FACT_TABLES_BATCH + FACT_TABLES_EVENT + STREAM_TABLES

catalog = []  # Will hold schema info for every table
data_frames = {}  # Cache DataFrames for downstream profiling
injection_warnings = []  # ZT: Collect injection scan results

# Use _FS_CSV_PATH (file:-prefixed in Fabric) for reliable Spark reads
_SPARK_CSV_BASE = _FS_CSV_PATH if _HAS_NOTEBOOKUTILS else CSV_BASE_PATH

print(f"Scanning {len(ALL_DISCOVERED)} CSV files from {CSV_BASE_PATH}...\n")

for table_name in ALL_DISCOVERED:
    csv_path = f"{_SPARK_CSV_BASE}/{table_name}.csv"
    try:
        df = (spark.read
              .option("header", True)
              .option("inferSchema", True)
              .option("multiLine", True)
              .csv(csv_path))

        # ZT: Validate all column names
        safe_cols = sanitize_columns([f.name for f in df.schema.fields])
        unsafe_count = len(df.schema.fields) - len(safe_cols)
        if unsafe_count > 0:
            log_audit_event("COLUMN_VALIDATION", table_name,
                f"{unsafe_count} unsafe column(s) detected", "WARNING")

        # ZT: Scan for CSV injection patterns in string data
        csv_warnings = check_csv_injection_risks(df, table_name)
        injection_warnings.extend(csv_warnings)

        # ZT: Flag sensitive/PII columns
        all_col_names = [f.name for f in df.schema.fields]
        _, sensitive_cols = filter_sensitive_columns(all_col_names)
        if sensitive_cols:
            log_audit_event("PII_DETECTED", table_name,
                f"Sensitive columns: {sensitive_cols}", "INFO")

        row_count = df.count()
        col_count = len(df.columns)
        data_frames[table_name] = df

        # Classify target
        if table_name in DIM_TABLES:
            target = "Lakehouse + Warehouse"
            category = "Dimension"
        elif table_name in FACT_TABLES_BATCH:
            target = "Lakehouse + Warehouse"
            category = "Fact (Batch)"
        elif table_name in FACT_TABLES_EVENT:
            target = "Eventhouse + Warehouse"
            category = "Fact (Event)"
        else:
            target = "Eventhouse"
            category = "Streaming"

        # Extract column info
        columns_info = [(f.name, str(f.dataType), f.nullable) for f in df.schema.fields]

        catalog.append({
            "table": table_name,
            "category": category,
            "target": target,
            "rows": row_count,
            "columns": col_count,
            "schema": columns_info,
            "status": "OK",
            "sensitive_columns": sensitive_cols,
        })
        log_audit_event("TABLE_SCANNED", table_name, f"{row_count} rows, {col_count} cols → {target}")
        print(f"  ✓ {table_name:<45} {category:<15} {row_count:>6} rows  {col_count:>3} cols → {target}")

    except Exception as e:
        catalog.append({
            "table": table_name,
            "category": "Unknown",
            "target": "N/A",
            "rows": 0,
            "columns": 0,
            "schema": [],
            "status": f"ERROR: {e}",
            "sensitive_columns": [],
        })
        log_audit_event("TABLE_SCAN_FAILED", table_name, str(e), "ERROR")
        print(f"  ✗ {table_name:<45} ERROR: {e}")

total_rows = sum(c["rows"] for c in catalog)
total_tables = len([c for c in catalog if c["status"] == "OK"])

# ZT: Report injection warnings
if injection_warnings:
    print(f"\n⚠ ZT SECURITY: {len(injection_warnings)} potential CSV injection risk(s) found:")
    for w in injection_warnings[:10]:  # Show first 10
        print(f"  • {w['table']}.{w['column']}: {w['issue']} — sample: {w['sample']!r}")
    if len(injection_warnings) > 10:
        print(f"  ... and {len(injection_warnings) - 10} more")

print(f"\n{'═'*70}")
print(f"Total: {total_tables} tables  |  {total_rows:,} rows  |  {len(catalog) - total_tables} errors")
print(f"{'═'*70}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# DATA PROFILING — Null counts, distinct values, numeric ranges
# ════════════════════════════════════════════════════════════════════════

quality_issues = []
profile_results = []

print("Profiling tables for data quality...\n")

for table_name, df in data_frames.items():
    row_count = df.count()
    if row_count == 0:
        quality_issues.append({"table": table_name, "issue": "Empty table (0 rows)", "severity": "HIGH"})
        continue

    for field in df.schema.fields:
        col = field.name
        null_count = df.where(F.col(col).isNull()).count()
        null_pct = round(null_count * 100.0 / row_count, 1)
        distinct_count = df.select(col).distinct().count()

        profile_entry = {
            "table": table_name,
            "column": col,
            "type": str(field.dataType),
            "nulls": null_count,
            "null_pct": null_pct,
            "distinct": distinct_count,
        }

        # Numeric columns: add min/max
        if isinstance(field.dataType, NumericType):
            stats = df.select(F.min(col), F.max(col)).first()
            profile_entry["min"] = stats[0]
            profile_entry["max"] = stats[1]

        profile_results.append(profile_entry)

        # Flag quality issues
        if null_pct > 50:
            quality_issues.append({"table": table_name, "column": col, "issue": f"{null_pct}% nulls", "severity": "HIGH"})
        elif null_pct > 20:
            quality_issues.append({"table": table_name, "column": col, "issue": f"{null_pct}% nulls", "severity": "MEDIUM"})

        # Flag potential ID columns with low cardinality
        if col.endswith('_id') and distinct_count < 2 and row_count > 5:
            quality_issues.append({"table": table_name, "column": col, "issue": f"ID column has only {distinct_count} distinct values", "severity": "MEDIUM"})

print(f"Profiled {len(profile_results)} columns across {len(data_frames)} tables.")
print(f"Found {len(quality_issues)} quality issues.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# DATA QUALITY REPORT
# ════════════════════════════════════════════════════════════════════════

if quality_issues:
    qi_df = pd.DataFrame(quality_issues)
    print(f"\n{'═'*70}")
    print(f"DATA QUALITY ISSUES ({len(quality_issues)})")
    print(f"{'═'*70}")
    for sev in ["HIGH", "MEDIUM", "LOW"]:
        subset = qi_df[qi_df["severity"] == sev]
        if len(subset) > 0:
            print(f"\n  [{sev}] — {len(subset)} issues")
            for _, row in subset.iterrows():
                col_str = f".{row.get('column', '')}" if 'column' in row and pd.notna(row.get('column')) else ''
                print(f"    • {row['table']}{col_str}: {row['issue']}")
else:
    print("\n✅ No data quality issues found!")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# DATA CATALOG — Full schema reference for all tables
# ════════════════════════════════════════════════════════════════════════

print(f"\n{'═'*80}")
print(f"DATA CATALOG — {INDUSTRY.upper()}")
print(f"{'═'*80}")

for entry in catalog:
    if entry["status"] != "OK":
        continue
    print(f"\n┌{'─'*78}┐")
    print(f"│ {entry['table']:<50} {entry['category']:<12} {entry['rows']:>5} rows │")
    print(f"│ Target: {entry['target']:<68} │")
    print(f"├{'─'*78}┤")
    for col_name, col_type, nullable in entry['schema']:
        null_str = "nullable" if nullable else "NOT NULL"
        print(f"│   {col_name:<35} {col_type:<25} {null_str:<12} │")
    print(f"└{'─'*78}┘")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# LOAD PLAN SUMMARY — What goes where
# ════════════════════════════════════════════════════════════════════════

summary_data = []
for entry in catalog:
    if entry["status"] != "OK":
        continue
    summary_data.append({
        "Table": entry["table"],
        "Category": entry["category"],
        "Rows": entry["rows"],
        "Columns": entry["columns"],
        "→ Lakehouse": "✓" if entry["table"] in LAKEHOUSE_TABLES else "—",
        "→ Warehouse": "✓" if entry["table"] in WAREHOUSE_TABLES else "—",
        "→ Eventhouse": "✓" if entry["table"] in EVENTHOUSE_TABLES else "—",
    })

summary_df = pd.DataFrame(summary_data)
print(f"\n{'═'*90}")
print(f"LOAD PLAN — {INDUSTRY.upper()}")
print(f"{'═'*90}")
print(summary_df.to_string(index=False))
print(f"\nLakehouse:  {summary_df['→ Lakehouse'].value_counts().get('✓', 0)} tables")
print(f"Warehouse:  {summary_df['→ Warehouse'].value_counts().get('✓', 0)} tables")
print(f"Eventhouse: {summary_df['→ Eventhouse'].value_counts().get('✓', 0)} tables")
print(f"\n✅ Ingestion scan complete. Proceed to loading notebooks.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# COMPLETE PIPELINE STEP
# ════════════════════════════════════════════════════════════════════════

try:
    total_tables = len(catalog)
    total_rows = sum(t.get("rows", 0) for t in catalog if isinstance(t, dict))
    quality_issues_count = len(quality_issues) if 'quality_issues' in dir() else 0
    log_pipeline_event("DATA_PROFILED", f"{total_tables} tables", 
        f"{total_rows:,} rows, {quality_issues_count} quality issues")
    pipeline_step_complete("01_Data_Ingestion", f"Profiled {total_tables} tables with {total_rows:,} rows")
except NameError:
    pass